In [1]:
import sys 
import torch

# 1. CRITICAL: Do the SQLite swap BEFORE importing ChromaDB or LangChain
try:
    __import__('pysqlite3')
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass # Fallback in case pysqlite3 isn't installed and standard sqlite3 is fine

import os
import json
import re
import pandas as pd
from tqdm import tqdm

# Now it is safe to import ChromaDB
import chromadb
from transformers import T5ForConditionalGeneration, T5Tokenizer, pipeline, logging
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from groq import Groq

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub")
logging.set_verbosity_error()

pd.set_option('display.max_colwidth', None)

device = 'cpu'

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large").to(device)

def call_llm(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(device) # Move inputs to the same device as the model
    
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Load data
train = pd.read_csv("./data/claims/train.csv")
test = pd.read_csv("./data/claims/test.csv")

print("done")

/media/krishiv/New Volume/ai/ai club dc/mini_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 558/558 [00:00<00:00, 17130.51it/s]


done


In [2]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 80,
    separators = ["\n\n", "\n", ". ", " ", ""]
)

In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device' : 'cpu'}, 
)

import json
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def build_story_db(story_path: str, base_dir: str = "./vectordb"):
    story_name = os.path.splitext(os.path.basename(story_path))[0]
    db_path = os.path.join(base_dir, story_name)
    os.makedirs(db_path, exist_ok=True)

    with open(story_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    chunks = splitter.split_text(raw_text)
    with open(os.path.join(db_path, "chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)

    Chroma.from_texts(
        texts=chunks,
        embedding=embedding_model,
        persist_directory=db_path,
    )
    print(f"✓ Built DB for '{story_name}' → {db_path}  ({len(chunks)} chunks)")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11135.80it/s]


In [4]:
sample_story = "./data/sample_story.txt"
book1_path = "./data/story/In search of the castaways.txt"
book2_path = "./data/story/The Count of Monte Cristo.txt"

build_story_db(sample_story)
build_story_db(book1_path)
build_story_db(book2_path)

✓ Built DB for 'sample_story' → ./vectordb/sample_story  (5 chunks)
✓ Built DB for 'In search of the castaways' → ./vectordb/In search of the castaways  (3068 chunks)
✓ Built DB for 'The Count of Monte Cristo' → ./vectordb/The Count of Monte Cristo  (9646 chunks)


In [4]:
_db_cache = {}   # story_name → Chroma
_chunk_cache = {}   # story_name → list[str]
_bm25_cache = {}   # story_name → BM25Okapi
_cross_encoder = None

def get_story_db(story_name, base_dir="./vectordb"):
    if story_name not in _db_cache:
        _db_cache[story_name] = Chroma(
            persist_directory=os.path.join(base_dir, story_name),
            embedding_function=embedding_model,
        )
    return _db_cache[story_name]

In [5]:
def _tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def get_story_chunks(story_name, base_dir="./vectordb"):
    if story_name not in _chunk_cache:
        chunks_path = os.path.join(base_dir, story_name, "chunks.json")
        with open(chunks_path, "r", encoding="utf-8") as f:
            _chunk_cache[story_name] = json.load(f)
    return _chunk_cache[story_name]


def get_story_bm25(story_name, base_dir="./vectordb"):
    if story_name not in _bm25_cache:
        chunks = get_story_chunks(story_name, base_dir)
        tokenized_chunks = [_tokenize(chunk) for chunk in chunks]
        _bm25_cache[story_name] = BM25Okapi(tokenized_chunks)
    return _bm25_cache[story_name]


def get_cross_encoder():
    global _cross_encoder
    if _cross_encoder is None:
        _cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
    return _cross_encoder


def retrieve_chunks(query, story_name, k=4, base_dir="./vectordb", k_candidates=20, rerank_top_n=10):
    vdb = get_story_db(story_name, base_dir)
    all_chunks = get_story_chunks(story_name, base_dir)
    bm25 = get_story_bm25(story_name, base_dir)
    cross_encoder = get_cross_encoder()

    vector_docs = vdb.similarity_search(query, k=min(k_candidates, len(all_chunks)))
    vector_ranked = [doc.page_content for doc in vector_docs]

    query_tokens = _tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k_candidates]
    bm25_ranked = [all_chunks[i] for i in bm25_indices]

    scores = {}
    for ranked in (vector_ranked, bm25_ranked):
        for rank, chunk in enumerate(ranked):
            scores[chunk] = scores.get(chunk, 0.0) + 1.0 / (rank + 60)

    fused = [chunk for chunk, _ in sorted(scores.items(), key=lambda item: item[1], reverse=True)][:k_candidates]

    rerank_pool = fused[:min(rerank_top_n, len(fused))]
    if rerank_pool:
        pairs = [(query, chunk) for chunk in rerank_pool]
        ce_scores = cross_encoder.predict(pairs)
        reranked = sorted(zip(rerank_pool, ce_scores), key=lambda item: item[1], reverse=True)
        fused = [chunk for chunk, _ in reranked] + fused[len(rerank_pool):]

    seen = set()
    unique = []
    for chunk in fused:
        if chunk not in seen:
            seen.add(chunk)
            unique.append(chunk)
        if len(unique) == k:
            break
    return unique


def retrieve_context_simple(query, story_name, k=4, base_dir="./vectordb"):
    return retrieve_chunks(query, story_name, k=k, base_dir=base_dir)


def search_story(query, story_name, k=4, base_dir="./vectordb"):
    return retrieve_chunks(query, story_name, k=k, base_dir=base_dir)

In [6]:
def build_standard_prompt(claim, context_chunks):
    context_str = "\n\n".join([f"[Passage {i+1}] : {chunk}" for i, chunk in enumerate(context_chunks)])
    prompt = f"""You are a story fact-checker. Carefully read the passages below and decide if the claim is consistent with them.

    story passages : 
    {context_str}

    claim :
    "{claim}"

    is this claim consistent with the story passages? answer with only '1' for consistent or '0' for inconsistent.
    Don't answer with "no" or "yes". use only 1 and 0. output '-1' if unsure/undecided.
    Answer : """

    return prompt

In [7]:
def build_cot_prompt(claim, context_chunks):
    context_str = "\n\n".join([f"[Passage {i+1}] : {chunk}" for i, chunk in enumerate(context_chunks)])
    prompt = f"""You are a story fact-checker. You must reason briefly then give a verdict.

STORY PASSAGES:
{context_str}

CLAIM TO VERIFY: "{claim}"

Follow these steps. Keep each step to ONE sentence only:

Step 1 — What do the passages say about this topic? (one sentence)
Step 2 — What does the claim say? (one sentence)
Step 3 — Do they match or contradict? (one sentence)
Step 4 — VERDICT: 1 (consistent) or VERDICT: 0 (inconsistent)

Step 1: 
Step 2: 
Step 3: 
VERDICT: 
"""

    return prompt


In [8]:
def parse_verdict(llm_response):
    response = llm_response.strip().lower()
    if not response:
        return -1

    verdict_tokens = {"1", "yes", "true", "supported", "consistent"}
    negative_tokens = {"0", "no", "false", "unsupported", "inconsistent", "contradict", "contradiction", "contradicts"}

    for line in reversed(response.splitlines()):
        token = line.strip().strip("\"'").lower()
        if token in verdict_tokens:
            return 1
        if token in negative_tokens:
            return 0

    if response in verdict_tokens:
        return 1
    if response in negative_tokens:
        return 0

    if response.startswith("1") or response.startswith("yes") or response.startswith("true"):
        return 1
    if response.startswith("0") or response.startswith("no") or response.startswith("false"):
        return 0

    if "consistent" in response and "inconsistent" not in response:
        return 1
    if "inconsistent" in response or "contradict" in response or "contradiction" in response:
        return 0

    return -1


import re
from typing import Optional


_VERDICT_LINE_RE = re.compile(
    r"(?im)^\s*(?:final\s+verdict|verdict|answer)\s*[:\-]\s*([01]|yes|no|true|false|supported|unsupported|consistent|inconsistent|contradict(?:s|ion)?)\s*$"
)
_STEP4_LINE_RE = re.compile(r"(?im)^\s*step\s*4\b.*?([01]|yes|no|true|false|supported|unsupported|consistent|inconsistent|contradict(?:s|ion)?)\s*$")
_STANDALONE_VERDICT_LINE_RE = re.compile(r"(?im)^\s*([01]|yes|no|true|false|supported|unsupported|consistent|inconsistent|contradict(?:s|ion)?)\s*$")
_BRACKETED_DIGIT_RE = re.compile(r"""(?x)
    (?:\[\s*['"]?([01]|yes|no|true|false)['"]?\s*\]) |
    (?:['"]([01]|yes|no|true|false)['"])
""")


def _parse_binary_verdict(text: str, *, undecided: int = -1) -> int:
    """
    Robustly parse a binary verdict from model output.

    Returns:
      1 -> consistent
      0 -> inconsistent / contradict
     -1 -> undecided (default)
    """
    if not isinstance(text, str) or not text.strip():
        return undecided

    def _normalize(token):
        token = token.strip().lower()
        if token in {"1", "yes", "true", "supported", "consistent"}:
            return 1
        if token in {"0", "no", "false", "unsupported", "inconsistent", "contradict", "contradiction", "contradicts"}:
            return 0
        return None

    matches = list(_VERDICT_LINE_RE.finditer(text))
    if matches:
        verdict = _normalize(matches[-1].group(1))
        if verdict is not None:
            return verdict

    matches = list(_STEP4_LINE_RE.finditer(text))
    if matches:
        verdict = _normalize(matches[-1].group(1))
        if verdict is not None:
            return verdict

    matches = list(_STANDALONE_VERDICT_LINE_RE.finditer(text))
    if matches:
        verdict = _normalize(matches[-1].group(1))
        if verdict is not None:
            return verdict

    matches = list(_BRACKETED_DIGIT_RE.finditer(text))
    if matches:
        for match in reversed(matches):
            token = next((group for group in match.groups() if group), None)
            verdict = _normalize(token) if token else None
            if verdict is not None:
                return verdict

    lower = text.lower()
    has_consistent = "consistent" in lower
    has_inconsistent = "inconsistent" in lower or "contradict" in lower or "contradiction" in lower
    if has_consistent and not has_inconsistent:
        return 1
    if has_inconsistent and not has_consistent:
        return 0

    if lower.startswith(("yes", "true", "supported")):
        return 1
    if lower.startswith(("no", "false", "unsupported")):
        return 0

    return undecided


def parse_verdict(text: str, *, undecided: int = -1) -> int:
    return _parse_binary_verdict(text, undecided=undecided)


def parse_verdict_cot(text: str, *, undecided: int = -1) -> int:
    return _parse_binary_verdict(text, undecided=undecided)

In [9]:
def verify_claim(claim, story_name, k = 3, prompt_type = "standard", verbose = False):
    context_chunks = retrieve_context_simple(claim, story_name=story_name, k=k)

    if prompt_type == "cot":
        prompt = build_cot_prompt(claim, context_chunks)
        max_tokens = 600
    else :
        prompt = build_standard_prompt(claim, context_chunks)
        max_tokens = 20

    if verbose:
        print("=" * 60)
        print(f"[{prompt_type.upper()}] CLAIM: {claim}")
        print("-" * 60)

    raw_response = call_llm(prompt, max_new_tokens=max_tokens)

    if verbose:
        print(f"RAW RESPONSE:\n{raw_response}")

    if prompt_type == "cot":
        verdict = parse_verdict_cot(raw_response)
    else:
        verdict = parse_verdict(raw_response)

    if verbose:
        verdict_label = {1: "CONSISTENT ✅", 0: "INCONSISTENT ❌", -1: "UNDECIDED ⚠️"}
        print(f"\nVERDICT: {verdict_label.get(verdict, '?')}")
        print("=" * 60)

    return {
        "claim": claim,
        "verdict": verdict,
        "raw_response": raw_response,
        "context": context_chunks
    }

print("verify_claim updated with CoT support.") 

verify_claim updated with CoT support.


In [10]:
result_1 = verify_claim(
    claim="Elara grew up in an orphanage.",
    verbose = True,
    story_name = "sample_story",
    prompt_type = "cot",
)

result_2 = verify_claim(
    claim="Elara grew up with wealthy parents",
    story_name = "sample_story",
    verbose = True,
    prompt_type = "cot",
)

Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 5497.46it/s]


[COT] CLAIM: Elara grew up in an orphanage.
------------------------------------------------------------
RAW RESPONSE:
1

VERDICT: CONSISTENT ✅
[COT] CLAIM: Elara grew up with wealthy parents
------------------------------------------------------------
RAW RESPONSE:
0

VERDICT: INCONSISTENT ❌


In [12]:
BOOK_NAME_MAP = {
    "In Search of the Castaways": "In search of the castaways",
    "The Count of Monte Cristo":  "The Count of Monte Cristo",
}

def predict_on_csv(
    csv_path: str,
    n_samples,
    prompt_type: str = "standard",
    output_path: str = None,
) -> pd.DataFrame:
    """
    Runs verify_claim on every row in a claims CSV (train.csv or test.csv).

    Expected CSV columns: id, book_name, char, caption, content, label
      - 'content'   → the claim text to verify
      - 'book_name' → used to select the correct story vector DB
      - 'label'     → ground-truth (consistent / contradict), used for accuracy if present

    Returns a DataFrame with predictions appended.
    """
    df = pd.read_csv(csv_path)
    if n_samples:
        df = df.head(n_samples)

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Predicting [{prompt_type}]"):
        claim      = row["content"]
        book_name  = row["book_name"]
        story_name = BOOK_NAME_MAP.get(book_name, book_name)  # fallback to raw name

        result = verify_claim(claim, story_name=story_name, prompt_type=prompt_type)

        rows.append({
            "id":            row["id"],
            "book_name":     book_name,
            "claim":         claim,
            "ground_truth":  row.get("label", None),   # None if test.csv has no labels
            "predicted_int": result["verdict"],          # 1 / 0 / -1
            "raw_response":  result["raw_response"],
            "context" : result["context"],
        })

    results_df = pd.DataFrame(rows)

    # ── Accuracy report (only if ground-truth labels exist) ───────────────────
    if "label" in df.columns:
        # map string labels to int for comparison
        label_map = {"consistent": 1, "contradict": 0}
        results_df["gt_int"] = results_df["ground_truth"].map(label_map)
        decided = results_df[results_df["predicted_int"] != -1]
        correct = (decided["predicted_int"] == decided["gt_int"]).sum()
        if len(decided):
            print(f"\nAccuracy: {correct}/{len(decided)}  = {correct/len(decided)*100:.1f}%")
        else:
            print("\nAccuracy: n/a (no decided predictions)")
        print(f"Undecided: {len(results_df) - len(decided)}")

    results_df = results_df.drop(['ground_truth'], axis = 1)

    if output_path:
        results_df.to_csv(output_path, index=False)
        print(f"Saved to {output_path}")

    return results_df

In [13]:
train_results = predict_on_csv(
    csv_path="./data/claims/train.csv",
    prompt_type="cot",
    output_path="./data/claims/train_predictions.csv",
    n_samples = 20,
)

Predicting [cot]: 100%|█████████████████████████| 20/20 [00:24<00:00,  1.21s/it]


Accuracy: 11/20  = 55.0%
Undecided: 0
Saved to ./data/claims/train_predictions.csv


In [14]:
t = train_results
print(t)

     id                   book_name  \
0    46  In Search of the Castaways   
1   137   The Count of Monte Cristo   
2    74  In Search of the Castaways   
3   109   The Count of Monte Cristo   
4   104   The Count of Monte Cristo   
5    35  In Search of the Castaways   
6    18  In Search of the Castaways   
7    31  In Search of the Castaways   
8    68  In Search of the Castaways   
9     9  In Search of the Castaways   
10   84  In Search of the Castaways   
11   83  In Search of the Castaways   
12  134   The Count of Monte Cristo   
13   88   The Count of Monte Cristo   
14   67  In Search of the Castaways   
15  112   The Count of Monte Cristo   
16   79  In Search of the Castaways   
17   55  In Search of the Castaways   
18   12  In Search of the Castaways   
19   13  In Search of the Castaways   

                                                                                                                                                                                    

In [13]:
# search_story is defined in the shared retrieval helper cell.


def build_agent_prompt(claim, context_chunks, attempt=1):
    context_str = "\n\n".join([f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)])
    return f"""You are a story fact-checker.

Use the passages to decide whether the claim is supported.

STORY PASSAGES:
{context_str}

CLAIM: "{claim}"

Return exactly one token:
1 = supported / consistent
0 = contradicted / inconsistent
NEED_MORE_INFO = only if the passages are clearly insufficient

If you choose NEED_MORE_INFO, add a second line:
FOLLOW_UP_QUERY: <short search query>
"""


def build_agent_verdict_prompt(claim, context_chunks):
    context_str = "\n\n".join([f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)])
    return f"""You are a story fact-checker.

Use the passages to decide if the claim is supported.

STORY PASSAGES:
{context_str}

CLAIM: "{claim}"

Return exactly one token: 1 or 0.
Do not explain.
"""


def parse_agent_response(response):
    import json as _json

    need_more_info = False
    follow_up_query = ""
    verdict = -1
    text = response.strip()
    lower = text.lower()

    if text.startswith("{") and text.endswith("}"):
        try:
            payload = _json.loads(text)
            need_more_info = bool(payload.get("need_more_info", False))
            follow_up_query = str(payload.get("follow_up_query", "") or "").strip()
            raw_verdict = payload.get("verdict", None)
            if raw_verdict in {0, 1}:
                verdict = int(raw_verdict)
            elif isinstance(raw_verdict, str) and raw_verdict.strip() in {"0", "1", "yes", "no", "true", "false"}:
                verdict = 1 if raw_verdict.strip().lower() in {"1", "yes", "true"} else 0
        except Exception:
            pass

    if verdict == -1:
        if "need_more_info" in lower or "need more info" in lower or "follow_up_query" in lower:
            need_more_info = True
        if "follow_up_query:" in lower:
            for line in text.splitlines():
                stripped = line.strip()
                if stripped.lower().startswith("follow_up_query:"):
                    follow_up_query = stripped.split(":", 1)[-1].strip()

        verdict_tokens = {"1", "yes", "true", "supported", "consistent"}
        negative_tokens = {"0", "no", "false", "unsupported", "inconsistent", "contradict", "contradiction", "contradicts"}
        for line in reversed(text.splitlines()):
            token = line.strip().strip("\"'").lower()
            if token in verdict_tokens:
                verdict = 1
                break
            if token in negative_tokens:
                verdict = 0
                break

        if verdict == -1:
            if lower.startswith(tuple(verdict_tokens)):
                verdict = 1
            elif lower.startswith(tuple(negative_tokens)):
                verdict = 0

        if verdict == -1:
            if "consistent" in lower and "inconsistent" not in lower:
                verdict = 1
            elif "inconsistent" in lower or "contradict" in lower or "contradiction" in lower:
                verdict = 0

    return {
        "need_more_info": need_more_info,
        "follow_up_query": follow_up_query,
        "verdict": verdict,
    }


def build_follow_up_query(claim, context_chunks):
    stopwords = {
        "the", "a", "an", "and", "or", "to", "of", "in", "on", "for", "with", "by", "at", "from",
        "his", "her", "their", "he", "she", "it", "they", "was", "were", "is", "are", "be", "been",
        "had", "has", "have", "as", "this", "that", "these", "those", "after", "before", "when", "while",
        "who", "whom", "whose", "what", "which", "where", "why", "how", "into", "over", "under", "about",
        "up", "down", "out", "off", "than", "then", "there", "here", "also", "very", "more", "most",
    }
    tokens = []
    for source_text in [claim] + list(context_chunks):
        for token in re.findall(r"[A-Za-z0-9']+", str(source_text).lower()):
            if token not in stopwords and len(token) > 2:
                tokens.append(token)
    if not tokens:
        return claim
    return " ".join(tokens[:10])


def verify_claim_agent(claim, story_name, max_rounds=2, verbose=False):
    """Agentic loop: search, read, decide — or search again."""
    all_context = []
    query = claim
    log = []
    response = ""
    followup_mode = False

    for round_num in range(1, max_rounds + 1):
        new_chunks = search_story(query, story_name)
        all_context.extend(new_chunks)

        seen = set()
        unique_context = []
        for chunk in all_context:
            if chunk not in seen:
                seen.add(chunk)
                unique_context.append(chunk)
        all_context = unique_context

        prompt = build_agent_verdict_prompt(claim, all_context) if followup_mode else build_agent_prompt(claim, all_context, attempt=round_num)
        response = call_llm(prompt, max_new_tokens=96)
        parsed = parse_agent_response(response)

        log.append({
            "round": round_num,
            "query": query,
            "chunks_retrieved": len(new_chunks),
            "total_context": len(all_context),
            "need_more_info": parsed["need_more_info"],
            "follow_up_query": parsed["follow_up_query"],
            "verdict": parsed["verdict"],
            "response": response,
        })

        if verbose:
            print(f"--- Round {round_num} ---")
            print(f"Query: {query}")
            print(f"Response: {response}")

        if parsed["need_more_info"]:
            query = parsed["follow_up_query"] or build_follow_up_query(claim, all_context)
            followup_mode = True
            continue

        verdict = parsed["verdict"] if parsed["verdict"] in {0, 1} else parse_verdict(response)
        return {
            "claim": claim,
            "verdict": verdict,
            "raw_response": response,
            "context": all_context,
            "rounds": round_num,
            "log": log,
        }

    verdict = parse_verdict(response)
    return {
        "claim": claim,
        "verdict": verdict,
        "raw_response": response,
        "context": all_context,
        "rounds": max_rounds,
        "log": log,
    }


In [14]:
def predict_claim_with_agent(claim, story_name, max_rounds=2, verbose=True):
    result = verify_claim_agent(
        claim=claim,
        story_name=story_name,
        max_rounds=max_rounds,
        verbose=verbose,
    )
    label_map = {1: "consistent", 0: "contradict", -1: "undecided"}
    result["predicted_label"] = label_map.get(result["verdict"], "undecided")
    return result


def print_agent_trace(result):
    print(f"Claim: {result['claim']}")
    print(f"Verdict: {result['verdict']} ({result.get('predicted_label', 'undecided')})")
    print(f"Rounds: {result['rounds']}")
    print("Trace:")
    for step in result["log"]:
        print(
            f"  Round {step['round']}: query={step['query']!r}, chunks={step['chunks_retrieved']}, "
            f"need_more_info={step['need_more_info']}, verdict={step['verdict']}, "
            f"follow_up_query={step['follow_up_query']!r}"
        )


def predict_on_csv_with_agent(csv_path, n_samples=20, output_path="./data/claims/train_predictions_agent.csv", max_rounds=2):
    df = pd.read_csv(csv_path)
    if n_samples:
        df = df.head(n_samples)
    label_map = {"consistent": 1, "contradict": 0}

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Predicting [agent]"):
        claim = row["content"]
        book_name = row["book_name"]
        story_name = BOOK_NAME_MAP.get(book_name, book_name)
        result = predict_claim_with_agent(claim, story_name, max_rounds=max_rounds, verbose=False)

        rows.append({
            "id": row["id"],
            "book_name": book_name,
            "char": row.get("char", ""),
            "caption": row.get("caption", ""),
            "claim": claim,
            "gt_int": label_map.get(row.get("label", None), None),
            "predicted_int": result["verdict"],
            "predicted_label": result["predicted_label"],
            "rounds": result["rounds"],
            "raw_response": result["raw_response"],
            "context": result["context"],
            "log": result["log"],
        })

    out_df = pd.DataFrame(rows)
    if "label" in df.columns:
        decided = out_df[out_df["predicted_int"].isin([0, 1])]
        if len(decided):
            correct = (decided["predicted_int"] == decided["gt_int"]).sum()
            print(f"Accuracy: {correct/len(decided):.2%} ({correct}/{len(decided)})")
        else:
            print("Accuracy: n/a (no decided predictions)")
        print(f"Undecided: {len(out_df) - len(decided)}")

    if output_path:
        out_df.to_csv(output_path, index=False)
        print(f"Saved to {output_path}")

    return out_df


agent_train_results = predict_on_csv_with_agent(
    csv_path="./data/claims/train.csv",
    n_samples=20,
    output_path="./data/claims/train_predictions_agent.csv",
    max_rounds=2,
)


Predicting [agent]: 100%|███████████████████████| 20/20 [00:26<00:00,  1.33s/it]

Accuracy: 65.00% (13/20)
Undecided: 0
Saved to ./data/claims/train_predictions_agent.csv
